Phase 1: Data Check

Phase 2: Building core screen engine

In [30]:
import pandas as pd
from fuzzywuzzy import fuzz

df_customers = pd.read_csv('D:/New Resumes/Latest/Projects/Sanctions Screening Project/Data/Customer_Dataset.csv')
df_ofac = pd.read_csv('D:/New Resumes/Latest/Projects/Sanctions Screening Project/Data/OFAC_SDN_List.csv')
df_pep = pd.read_csv('D:/New Resumes/Latest/Projects/Sanctions Screening Project/Data/PEP_List.csv')

# 1. Business Logic Rules
def get_risk_category(score):
    if score >= 95: return 'CRITICAL'
    elif score >= 80: return 'HIGH'
    elif score >= 75: return 'MEDIUM'
    else: return 'LOW'

def get_decision(score):
    if score >= 95: return 'BLOCK - Manual Review Required'
    elif score >= 85: return 'HOLD - Enhanced Verification'
    elif score >= 75: return 'REVIEW - Secondary Check'
    else: return 'APPROVE'

# 2. The Direct-to-DataFrame Screener
def screen_name(customer_name, df_ofac, df_pep):
    customer_name = str(customer_name).upper().strip()
    
    highest_score = 0
    best_match = 'None'
    source = 'None'
    sanc_prog = None
    pep_details = None
    
    # --- Check df_ofac (OFAC Sanctions) ---
    for _, row in df_ofac.iterrows():
        # Using 'sdn_name' and 'sdn_program' from df_ofac
        watch_name = str(row.get('sdn_name', '')).upper().strip()
        score = fuzz.token_sort_ratio(customer_name, watch_name)
        
        if score > highest_score:
            highest_score = score
            best_match = watch_name
            source = 'SANCTIONS'
            sanc_prog = row.get('sdn_program', 'Unknown')
            pep_details = None
            
    # --- Check df_pep (PEP List) ---
    for _, row in df_pep.iterrows():
        # Using 'full_name' from df_pep
        watch_name = str(row.get('full_name', '')).upper().strip()
        score = fuzz.token_sort_ratio(customer_name, watch_name)
        
        if score > highest_score:
            highest_score = score
            best_match = watch_name
            source = 'PEP'
            sanc_prog = None 
            # Safely grab context since df2 columns were cut off in the info output
            pep_details = row.get('country', row.get('organization', 'Unknown'))

    # Clean up low scores
    if highest_score < 75:
        best_match, source, sanc_prog, pep_details = 'None', 'None', None, None

    # Return as a series to map directly to our new columns
    return pd.Series([
        best_match, 
        highest_score, 
        source, 
        sanc_prog, 
        pep_details, 
        get_risk_category(highest_score), 
        get_decision(highest_score)
    ])


new_columns = [
    'matched_name', 
    'match_score', 
    'list_source', 
    'matched_sdn_program', 
    'matched_pep_details', 
    'screening_risk_level',
    'final_decision'
]


# Test on just the first 10 customers
final_df = df_customers.head(200).copy()

final_df[new_columns] = final_df['customer_name'].apply(
    lambda x: screen_name(x, df_ofac, df_pep)
)

print(final_df)

    customer_id                 customer_name       country date_of_birth  \
0     CUS141001  Juann Carlos Montoya Sanchez         Syria    28-08-1990   
1     CUS141002               Brigitte Macron        France    13-04-1953   
2     CUS141003                   Ivan Ivanov  Saudi Arabia    22-07-1948   
3     CUS141004              Svetlana Johnson  Saudi Arabia    31-07-1951   
4     CUS141005                    Nadia Park         Syria    02-12-1952   
..          ...                           ...           ...           ...   
195   CUS141196               Larissa Johnson        Turkey    06-01-1955   
196   CUS141197                   Miriam Chen        Canada    20-10-1977   
197   CUS141198                 Ivan Andersen        Canada    02-11-1983   
198   CUS141199                  Kwame Garcia        France    15-08-1999   
199   CUS141200                  Amina Tanaka       Germany    02-09-1968   

     gender                                 address  id_number customer_typ